In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/drive/MyDrive/Sheep_Project/dataset

In [ ]:
!mv /content/drive/MyDrive/Sheep_Teeth_Dataset.zip \
    /content/drive/MyDrive/Sheep_Project/dataset/

In [ ]:
!unzip /content/drive/MyDrive/Sheep_Project/dataset/Sheep_Teeth_Dataset.zip \
      -d /content/drive/MyDrive/Sheep_Project/dataset/

In [ ]:
import numpy as np
import tensorflow as tf
import random
import os

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
dataset_path = "/content/drive/MyDrive/Sheep_Project/dataset/Sheep_Teeth_Dataset"

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(224,224),
    batch_size=16
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(224,224),
    batch_size=16
)
class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
plt.figure(figsize=(9,9))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05)
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,

)

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc)+1)

plt.figure(figsize=(10,5))
plt.plot(epochs, acc, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(epochs, loss, 'bo-', label='Training Loss')
plt.plot(epochs, val_loss, 'ro-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
model.save("/content/drive/MyDrive/Sheep_Teeth_Model.keras")

In [ ]:
import os

model_path = "/content/drive/MyDrive/Sheep_Teeth_Model.keras"
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("/content/drive/MyDrive/Sheep_Teeth_Model.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("/content/drive/MyDrive/Sheep_Teeth_Model.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
import os

tflite_path = "/content/drive/MyDrive/Sheep_Teeth_Model.tflite"

size_bytes = os.path.getsize(tflite_path)

size_kb = size_bytes / 1024
size_mb = size_kb / 1024

print(f"Size: {size_bytes} bytes")
print(f"Size: {size_kb:.2f} KB")
print(f"Size: {size_mb:.2f} MB")